# Solana Sniper: leakage-audited reproduction

This is the public Kaggle companion to the fully executed reproduction notebook and public source repository:

- [Executed reproduction notebook](https://github.com/DaoyuanLi2816/solana-sniper-bot-reverse-engineering/blob/main/notebooks/solana_sniper_reproduction.ipynb)
- [Public repository](https://github.com/DaoyuanLi2816/solana-sniper-bot-reverse-engineering)
- [Experiment errata](https://github.com/DaoyuanLi2816/solana-sniper-bot-reverse-engineering/blob/main/reports/ERRATA.md)

Competition rows are deliberately not embedded or republished here. The linked notebook contains code and aggregate outputs, was executed end to end against privately staged authorized artifacts, and documents the required official-data layout.

## Decision boundary

For every deployment, `t_decision` is the token-deployment timestamp. Classifier and entry-selection features are restricted to the deployment transaction, metadata visible by that instant, and deployer history with `history_time < t_decision`. Trades, candles, realized PnL, outcomes, and future deployer activity are label or backtest inputs only. The final holdout was not evaluated.

In [1]:
from pprint import pprint

certificate = {
    "classification_sha256": "57af874b0768eaf43c54f04d698cda9e3c3e1d9bf3e1a25c7c69910ecbe8817f",
    "rows": 218_350,
    "unique_tokens": 218_350,
    "strict_history_violations": 0,
    "utc_clock_mismatches": 0,
    "standard_validation_pr_auc": 0.07028141318814668,
    "precision": 0.1203377902885292,
    "recall": 0.2011173184357542,
    "f1": 0.15057787561915245,
    "threshold": 0.9370987253131746,
    "final_holdout_evaluated": False,
}
assert certificate["rows"] == certificate["unique_tokens"]
assert certificate["strict_history_violations"] == 0
assert certificate["utc_clock_mismatches"] == 0
assert certificate["final_holdout_evaluated"] is False
pprint(certificate)

{'classification_sha256': '57af874b0768eaf43c54f04d698cda9e3c3e1d9bf3e1a25c7c69910ecbe8817f',
 'f1': 0.15057787561915245,
 'final_holdout_evaluated': False,
 'precision': 0.1203377902885292,
 'recall': 0.2011173184357542,
 'rows': 218350,
 'standard_validation_pr_auc': 0.07028141318814668,
 'strict_history_violations': 0,
 'threshold': 0.9370987253131746,
 'unique_tokens': 218350,
 'utc_clock_mismatches': 0}


## Execution falsification

The optimistic first-observed-trade proxy passed the predeclared criterion only at requested delay zero. Replacing that proxy with the target wallet's training-derived transaction position reduced population-weighted coverage to 57.03%. At median fees, weighted mean return remained +8.11%, but unweighted median return was -5.56%, hit rate was 43.53%, maximum drawdown was 91.96%, and the fixed-capital path crossed zero. We therefore reject the executable position-lag candidate and report the raw zero-slot result only as an upper bound.

In [2]:
import pandas as pd

pd.DataFrame(
    [
        {
            "entry": "target wallet, strict pre-holdout June",
            "mean_return": 0.107118,
            "median_return": 0.030496,
            "hit_rate": 0.566327,
            "max_drawdown": 0.068893,
            "decision": "descriptive target",
        },
        {
            "entry": "position-lag replica, median fee",
            "mean_return": 0.081088,
            "median_return": -0.055575,
            "hit_rate": 0.435341,
            "max_drawdown": 0.919625,
            "decision": "rejected",
        },
    ]
).set_index("entry")

,mean_return,median_return,hit_rate,max_drawdown,decision
entry,,,,,
"target wallet, strict pre-holdout June",0.107118,0.030496,0.566327,0.068893,descriptive target
"position-lag replica, median fee",0.081088,-0.055575,0.435341,0.919625,rejected


## Reproduce the evidence

Clone the public repository, obtain the competition files only through Kaggle's official channel, stage them under a private data root as documented in `notebooks/README.md`, set `SOLANA_SNIPER_DATA_ROOT`, and execute `notebooks/solana_sniper_reproduction.ipynb`. The notebook rechecks source hashes, schema, row uniqueness, UTC time features, strict history, time splits, operating point, 0/1/2-slot execution, transaction-position lag, fees, hit rate, solvency, and drawdown. `uv.lock` fixes the environment; `experiments/manifest.jsonl` retains parameters, hashes, code revisions, conclusions, and negative results.